# 🔎 Security Log Investigation

## Objective
Analyze a synthetic authentication log to identify suspicious authentication behavior and reconstruct a likely incident timeline.

**Analyst:** Supriyo Malik  
**Dataset:** Synthetic authentication data for cybersecurity training.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
print('Libraries loaded successfully.')

In [ ]:
df = pd.read_csv('../data/auth_log.csv')
print(f'Total events: {len(df)}')
df.head()

In [ ]:
df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)
df['source_ip'] = df['source_ip'].astype(str).str.strip()
df['username'] = df['username'].astype(str).str.strip().str.lower()
df['action'] = df['action'].astype(str).str.strip().str.lower()
df['result'] = df['result'].astype(str).str.strip().str.lower()
df['device'] = df['device'].astype(str).str.strip()
df = df.sort_values('timestamp').reset_index(drop=True)
print('Data normalization completed.')
df.head()

In [ ]:
print('Unique IP addresses:', df['source_ip'].nunique())
print('Unique accounts:', df['username'].nunique())
print('\nAuthentication results:')
print(df['result'].value_counts())
print('\nActions:')
print(df['action'].value_counts())

In [ ]:
failed = df[df['result'] == 'failed'].copy()
print(f'Total failed authentication events: {len(failed)}')
failed.groupby('source_ip').size().sort_values(ascending=False).to_frame('failed_attempts')

In [ ]:
spraying = (failed.groupby('source_ip')['username'].nunique().reset_index(name='unique_accounts_targeted'))
spraying = spraying[spraying['unique_accounts_targeted'] >= 3].sort_values('unique_accounts_targeted', ascending=False)
print('Potential password-spraying sources:')
spraying

In [ ]:
bruteforce = (failed.groupby(['source_ip','username']).size().reset_index(name='failed_attempts'))
bruteforce = bruteforce[bruteforce['failed_attempts'] >= 5].sort_values('failed_attempts', ascending=False)
print('Potential brute-force activity:')
bruteforce

In [ ]:
successes = df[(df['action'] == 'login') & (df['result'] == 'success')]
successes[['timestamp','source_ip','username','device']]

In [ ]:
suspicious_ip = '10.10.10.50'
df[df['source_ip'] == suspicious_ip][['timestamp','source_ip','username','action','result','device']]

In [ ]:
privileged_events = df[df['action'].isin(['access'])]
privileged_events[['timestamp','source_ip','username','action','result']]

In [ ]:
service_events = df[df['username'] == 'service_account']
service_events[['timestamp','source_ip','username','action','result']]

In [ ]:
timeline = df[df['source_ip'].isin(['185.220.101.10','10.10.10.50','203.0.113.45'])][['timestamp','source_ip','username','action','result']].sort_values('timestamp')
timeline

In [ ]:
event_counts = df.set_index('timestamp').resample('5min').size()
plt.figure(figsize=(12,5))
event_counts.plot(kind='bar')
plt.title('Authentication Events Over Time')
plt.xlabel('Time')
plt.ylabel('Number of Events')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Investigation Findings

## Finding 1 — Password Spraying
The source IP `185.220.101.10` attempted authentication against five different accounts within a short period. This behavior is consistent with a password-spraying pattern.

## Finding 2 — Brute Force
The source IP `10.10.10.50` generated five failed authentication attempts against `alice`. A successful login occurred shortly afterward.

## Finding 3 — Unusual Privileged Access
After the successful authentication, `alice` accessed VPN resources and subsequently accessed the admin panel. The log confirms the sequence but does not independently prove account compromise or unauthorized activity.

## Finding 4 — Service Account Activity
The `service_account` authenticated successfully from `203.0.113.45` and accessed database administration and configuration resources. Further investigation is required to determine whether this activity was authorized.

# Evidence vs Assumptions

## Direct Evidence
- Multiple accounts were targeted from `185.220.101.10`.
- Five failed login attempts were recorded against `alice`.
- A successful login occurred after those failures.
- `alice` accessed VPN resources.
- `alice` accessed the admin panel.
- `service_account` accessed database administration and configuration resources.

## Assumptions / Hypotheses
- The password-spraying activity was malicious.
- The `alice` account was compromised.
- The admin-panel access was unauthorized.
- The service account was compromised or misused.

These hypotheses require additional evidence before they can be confirmed.

# Recommended Response Actions

## Immediate
1. Review the `alice` account.
2. Reset credentials if compromise is suspected.
3. Revoke active sessions.
4. Review MFA activity.
5. Investigate admin-panel activity.
6. Review service-account activity.
7. Preserve relevant authentication and application logs.

## Detection Improvements
- Alert when one IP targets multiple accounts.
- Alert when repeated failures are followed by success.
- Monitor privileged access after suspicious authentication.
- Monitor service accounts for unusual activity.
- Centralize authentication logs in a SIEM.

## Preventive Controls
- Multi-factor authentication
- Authentication rate limiting
- Account throttling
- Strong password policies
- Least-privilege access
- Service-account restrictions
- Security monitoring

# Conclusion

The synthetic authentication dataset contains several suspicious patterns: password-spraying behavior from `185.220.101.10`, repeated authentication failures against `alice`, successful authentication following those failures, subsequent VPN/admin-panel access, and privileged service-account activity.

The available data establishes these events but does not independently prove malicious intent or account compromise. Additional endpoint, VPN, MFA, application, firewall, and identity-provider logs should be reviewed before making a final incident determination.

**Investigation Type:** Synthetic / Educational  
**Analyst:** Supriyo Malik